In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

#Data
df = pd.read_csv("../../data/geocoded_data/new_scores.csv")

# Keep only valid coordinates
df = df[df["ok"] == True]
df = df.dropna(subset=["lat", "lng"])

print(df.shape)
df.head()


In [ ]:
#Convert to GeoDataFrame + Load Block Groups

geometry = [Point(xy) for xy in zip(df["lng"], df["lat"])]

gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print(gdf.crs)


In [ ]:
import requests
import zipfile
import os

# URL for Georgia Block Groups
url = "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_13_bg.zip"

# Local paths
zip_path = "tl_2020_13_bg.zip"
extract_path = "tl_2020_13_bg"

# Download
response = requests.get(url)
with open(zip_path, "wb") as f:
    f.write(response.content)

print("Download complete")

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete")

In [ ]:
import geopandas as gpd

block_groups = gpd.read_file("tl_2020_13_bg/tl_2020_13_bg.shp")

print(block_groups.shape)
# block_groups.head()
block_groups = block_groups.to_crs(gdf.crs)

In [ ]:
block_groups.to_file("block_groups.gpkg", driver="GPKG")
block_groups = gpd.read_file("block_groups.gpkg")

In [ ]:
joined = gpd.sjoin(
    gdf,
    block_groups,
    how="left",
    predicate="within"
)

In [ ]:
print(joined.shape)
joined.head()

In [ ]:
joined["GEOID"].isna().sum()

In [ ]:
joined[["lat", "lng", "GEOID"]].head()

In [ ]:
joined.head()

In [ ]:
bg_year_agg = joined.groupby(["GEOID", "year"]).agg({
    "Number": "count",
    "severity_score": "mean",
    "resolution_score": "mean",
    "resolution_time_hours": "mean"
}).reset_index()

bg_year_agg = bg_year_agg.rename(columns={
    "Number": "complaint_count",
    "severity_score": "avg_severity",
    "resolution_score": "avg_resolution_score",
    "resolution_time_hours": "avg_resolution_time"
})

In [ ]:
issue_counts = (
    joined
    .groupby(["GEOID", "year", "issue_type"])
    .size()
    .reset_index(name="count")
)

issue_dict = (
    issue_counts
    .groupby(["GEOID", "year"])
    .apply(lambda x: dict(zip(x["issue_type"], x["count"])))
    .reset_index(name="issue_type_counts")
)

domain_counts = (
    joined
    .groupby(["GEOID", "year", "domain"])
    .size()
    .reset_index(name="count")
)


domain_dict = (
    domain_counts
    .groupby(["GEOID", "year"])
    .apply(lambda x: dict(zip(x["domain"], x["count"])))
    .reset_index(name="domain_counts")
)

bg_year_agg = bg_year_agg.merge(issue_dict, on=["GEOID", "year"], how="left")
bg_year_agg = bg_year_agg.merge(domain_dict, on=["GEOID", "year"], how="left")


bg_year_agg["issue_type_counts"] = bg_year_agg["issue_type_counts"].apply(
    lambda x: x if isinstance(x, dict) else {}
)

bg_year_agg["domain_counts"] = bg_year_agg["domain_counts"].apply(
    lambda x: x if isinstance(x, dict) else {}
)

In [ ]:
print(bg_year_agg.shape)
bg_year_agg.head()

In [ ]:
block_groups_proj = block_groups.to_crs(epsg=3857)

block_groups_proj["area_sq_km"] = block_groups_proj.geometry.area / 1e6

In [ ]:
bg_year_agg = bg_year_agg.merge(
    block_groups_proj[["GEOID", "area_sq_km"]],
    on="GEOID",
    how="left"
)

In [ ]:
bg_year_agg["complaint_density"] = (
    bg_year_agg["complaint_count"] / bg_year_agg["area_sq_km"]
)

In [ ]:
bg_year_agg.describe()

In [ ]:
bg_year_agg.columns

In [ ]:
# #Plot
# map_df = block_groups.merge(
#     bg_year_agg,   # or final_df if you want ACS features too
#     on="GEOID",
#     how="left"
# )

# import matplotlib.pyplot as plt

# years = sorted(map_df["year"].dropna().unique())

# for yr in years:
#     fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
#     data = map_df[map_df["year"] == yr]
    
#     data.plot(
#         column="avg_severity",   # change this to other metrics later
#         cmap="OrRd",
#         legend=True,
#         ax=ax
#     )
    
#     ax.set_title(f"Severity - {yr}")
#     ax.axis("off")
    
#     plt.show()

In [ ]:
# Extract county FIPS from GEOID
joined["county_fips"] = joined["GEOID"].str[2:5]

# Get unique counties
counties = joined["county_fips"].dropna().unique()

# Ensure proper formatting (important for API)
counties = [str(c).zfill(3) for c in counties]

print("Counties in dataset:", counties)
print("Number of counties:", len(counties))

In [ ]:
from census import Census
import pandas as pd

c = Census("fa7883b213fcc5047737a09467ed916145ae32fa")

variables = (
    "B19013_001E",
    "B01003_001E",
    "B25003_003E",
    "B25003_001E",
    "C17002_002E",

    "B02001_001E",  # total race population
    "B02001_002E",  # white
    "B02001_003E",  # black
    "B02001_005E"   # asian

)

years = [2021, 2022, 2023]  # ACS available years

acs_all = []

for year in years:
    print(f"Fetching ACS {year}")
    
    for county in counties:
        data = c.acs5.state_county_blockgroup(
            variables,
            state_fips="13",
            county_fips=county,
            blockgroup="*",
            year=year
        )
        
        # attach year to each row
        for row in data:
            row["acs_year"] = year
        
        acs_all.extend(data)

acs_df = pd.DataFrame(acs_all)
print(acs_df.columns)

In [ ]:
acs_df["GEOID"] = (
    acs_df["state"] +
    acs_df["county"] +
    acs_df["tract"] +
    acs_df["block group"]
)

print(acs_df.shape)

In [ ]:
# Rename important columns
acs_df = acs_df.rename(columns={
    "B19013_001E": "median_income",
    "B01003_001E": "population",
    "C17002_002E": "poverty_rate",

    "B02001_001E": "race_total",
    "B02001_002E": "white_pop",
    "B02001_003E": "black_pop",
    "B02001_005E": "asian_pop"
})

print("Before cleaning:", acs_df.shape)

print("Missing values:")
print(acs_df[[
    "population",
    "median_income",
    "poverty_rate"
]].isna().sum())

In [ ]:
print(acs_df.columns)

In [ ]:
# Convert to numeric
cols_to_numeric = [
    "median_income",
    "population",
    "B25003_003E",
    "B25003_001E",
    "poverty_rate",
    "race_total",
    "white_pop",
    "black_pop",
    "asian_pop"
]

for col in cols_to_numeric:
    acs_df[col] = pd.to_numeric(acs_df[col], errors="coerce")

# Keep only valid population
acs_df = acs_df.dropna(subset=["population"])
acs_df = acs_df[acs_df["population"] > 0]

acs_df = acs_df[acs_df["median_income"] > 0]

acs_df = acs_df.replace(-666666666, pd.NA)

# Derived feature
acs_df["pct_renters"] = (
    acs_df["B25003_003E"] / acs_df["B25003_001E"]
)

In [ ]:
acs_df["pct_white"] = acs_df["white_pop"] / acs_df["race_total"]
acs_df["pct_black"] = acs_df["black_pop"] / acs_df["race_total"]
acs_df["pct_asian"] = acs_df["asian_pop"] / acs_df["race_total"]

In [ ]:
acs_df.shape

In [ ]:
def map_acs_year(y):
    if y <= 2023:
        return y
    else:
        return 2023 

bg_year_agg["acs_year"] = bg_year_agg["year"].apply(map_acs_year)

acs_df = acs_df[acs_df["GEOID"].isin(bg_year_agg["GEOID"])]
print("Filtered ACS shape:", acs_df.shape)

In [ ]:
final_df = bg_year_agg.merge(
    acs_df,
    on=["GEOID", "acs_year"],
    how="left"
)

In [ ]:
final_df["complaints_per_1000"] = (
    final_df["complaint_count"] / final_df["population"] * 1000
)

In [ ]:
final_df = final_df.sort_values(["GEOID", "year"])

# final_df["income_change"] = final_df.groupby("GEOID")["median_income"].diff()
# final_df["severity_change"] = final_df.groupby("GEOID")["avg_severity"].diff()

In [ ]:
final_df

In [ ]:
analysis_df = final_df.dropna(subset=[
    "complaints_per_1000"
])

In [ ]:
analysis_df.to_csv("../../data/geocoded_data/composite_scores.csv", index=False)

In [ ]:
analysis_df.columns

In [ ]:
print(block_groups["GEOID"].head())
print(analysis_df["GEOID"].head())
print(len(block_groups["GEOID"].iloc[0]))
print(len(analysis_df["GEOID"].iloc[0]))

In [ ]:
valid_geoids = analysis_df["GEOID"].unique()

final_gdf = block_groups[
    block_groups["GEOID"].isin(valid_geoids)
].merge(
    analysis_df,
    on="GEOID",
    how="left"
)

In [ ]:
final_gdf

In [ ]:
years = sorted(final_gdf["year"].dropna().unique())
print(years)

In [ ]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

import os

os.makedirs("geojson_outputs", exist_ok=True)

for year in years:
    df_year = final_gdf[final_gdf["year"] == year].copy()
    
    # ensure one row per GEOID
    df_year = df_year.drop_duplicates(subset=["GEOID"])
    
    # fill missing values
    df_year = df_year.fillna(0)
    
    # optional: compute final score
    df_year["final_score"] = (
    0.4 * normalize(df_year["complaints_per_1000"]) +
    0.3 * normalize(df_year["avg_severity"]) +
    0.3 * normalize(df_year["avg_resolution_score"])
) * 100
    
    # save file
    output_path = f"geojson_outputs/map_{year}.geojson"
    df_year.to_file(output_path, driver="GeoJSON")
    
    print(f"Saved {output_path}")

In [ ]:
import geopandas as gpd
test = gpd.read_file("geojson_outputs/map_2023.geojson")
print(test.head())

In [ ]:
corr = analysis_df[[
    "complaints_per_1000",
    "avg_severity",
    "avg_resolution_time",
    "median_income",
    "poverty_rate",
    "pct_renters",
    "pct_black"
]].corr()

print(corr)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(corr, annot=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
import statsmodels.api as sm

X = analysis_df[[
    "median_income",
    "poverty_rate",
    "pct_renters"
]]

y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
map_df = block_groups.merge(
    final_df,
    on="GEOID",
    how="left"
)

map_df.plot(
    column="complaints_per_1000",
    legend=True,
    figsize=(10, 8)
)
plt.title("Complaint Density by Block Group")
plt.show()

In [ ]:
map_df.plot(
    column="median_income",
    legend=True,
    figsize=(10, 8)
)
plt.title("Median Income by Block Group")
plt.show()

In [ ]:
# Regression

In [ ]:
import statsmodels.api as sm

X = analysis_df[[
    "median_income",
    "poverty_rate",
    "pct_renters"
]]

y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
analysis_df["income_k"] = analysis_df["median_income"] / 1000

In [ ]:
X = analysis_df[["income_k", "pct_renters", "pct_black"]]
y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = analysis_df[["pct_renters", "pct_black"]]
y = analysis_df["complaints_per_1000"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = analysis_df[["income_k", "pct_renters", "pct_black"]]
y = analysis_df["avg_severity"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = analysis_df[["income_k", "pct_renters", "poverty_rate", "pct_black"]]
y = analysis_df["avg_resolution_score"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
analysis_df["race_group"] = pd.qcut(
    analysis_df["pct_black"],
    3,
    labels=["low_black", "mid_black", "high_black"]
)

In [ ]:
analysis_df["race_group"].value_counts()

In [ ]:
analysis_df.groupby("race_group")[[
    "complaints_per_1000",
    "avg_severity",
    "avg_resolution_time",
    "median_income",
    "pct_renters"
]].mean()

In [ ]:
import statsmodels.api as sm

for group in ["low_black", "mid_black", "high_black"]:
    
    subset = analysis_df[analysis_df["race_group"] == group]
    
    X = subset[["income_k", "pct_renters"]]
    y = subset["complaints_per_1000"]
    
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    print("\n====================")
    print(f"Group: {group}")
    print("====================")
    print(model.summary())

In [ ]:
############### For getting names - independent pipeline

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

#Data
df = pd.read_csv("../../data/geocoded_data/new_scores.csv")

# Keep only valid coordinates
df = df[df["ok"] == True]
df = df.dropna(subset=["lat", "lng"])

print(df.shape)
df.head()

#Convert to GeoDataFrame + Load Block Groups

geometry = [Point(xy) for xy in zip(df["lng"], df["lat"])]

gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print(gdf.crs)


C:\Users\shrey\AppData\Local\Temp\ipykernel_28172\2056401898.py:6: DtypeWarning: Columns (3,4,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/geocoded_data/new_scores.csv")


(442395, 32)
EPSG:4326


In [2]:
block_groups = gpd.read_file("tl_2020_13_bg/tl_2020_13_bg.shp")
block_groups = block_groups.to_crs(gdf.crs)

In [4]:
# load neighborhoods
neighborhoods = gpd.read_file("Atlanta_Neighborhoods.geojson")

# match CRS
neighborhoods = neighborhoods.to_crs(block_groups.crs)

# keep only needed columns
neighborhoods = neighborhoods[["NAME", "geometry"]]
neighborhoods = neighborhoods.rename(columns={"NAME": "neighborhood_name"})

In [5]:
bg_with_neighborhood = gpd.sjoin(
    block_groups,
    neighborhoods,
    how="left",
    predicate="intersects"
)

# remove duplicates
bg_with_neighborhood = bg_with_neighborhood.drop_duplicates(subset=["GEOID"])

# keep only needed columns
bg_with_neighborhood = bg_with_neighborhood[["GEOID", "neighborhood_name"]]

In [6]:
print(bg_with_neighborhood.neighborhood_name.isna)

<bound method Series.isna of 0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
       ... 
7441    NaN
7442    NaN
7443    NaN
7444    NaN
7445    NaN
Name: neighborhood_name, Length: 7446, dtype: object>


In [7]:
print("block_groups CRS:", block_groups.crs)
print("neighborhoods CRS:", neighborhoods.crs)

block_groups CRS: EPSG:4326
neighborhoods CRS: EPSG:4326


In [ ]:
print(neighborhoods.geometry.head())

In [8]:
print(bg_with_neighborhood["neighborhood_name"].isna().sum())

6942


In [9]:
bg_with_neighborhood = bg_with_neighborhood[
    bg_with_neighborhood["neighborhood_name"].notna()
]

In [10]:
bg_with_neighborhood["GEOID"] = bg_with_neighborhood["GEOID"].astype(str)

bg_with_neighborhood.to_csv("bg_with_neighborhood.csv", index=False)

print("Saved bg_with_neighborhood.csv")

Saved bg_with_neighborhood.csv


In [ ]:
import os
import json

geojson_folder = "geojson_outputs"  # change if needed

geojson_geoids = set()

for file in os.listdir(geojson_folder):
    if file.endswith(".geojson"):
        path = os.path.join(geojson_folder, file)

        with open(path) as f:
            data = json.load(f)

        for feature in data["features"]:
            geoid = feature["properties"].get("GEOID")
            if geoid:
                geojson_geoids.add(str(geoid))

print("Total GEOIDs in GeoJSON files:", len(geojson_geoids))

In [ ]:
bg_geoids = set(bg_with_neighborhood["GEOID"].astype(str))

print("Total GEOIDs in bg_with_neighborhood:", len(bg_geoids))

missing_in_bg = geojson_geoids - bg_geoids
print("GEOIDs in GeoJSON but NOT in bg_with_neighborhood:", len(missing_in_bg))

extra_in_bg = bg_geoids - geojson_geoids
print("GEOIDs in bg_with_neighborhood but NOT in GeoJSON:", len(extra_in_bg))

print("Sample missing GEOIDs:", list(missing_in_bg)[:10])
print("Sample extra GEOIDs:", list(extra_in_bg)[:10])

missing_names = bg_with_neighborhood["neighborhood_name"].isna().sum()

print("Block groups without neighborhood:", missing_names)

In [ ]:
check_geoids = [
    "131210085003","131210088022","131210080005","131210084001","131210070023",
    "131210077031","131210073013","131210055041","131210070021","131210063002",
    "130630403081","130890215021","130630402031","131210006022","131210106042",
    "130890229002","130890214161","130890214173","131210108023","130890238023"
]

In [ ]:
result = bg_with_neighborhood[
    bg_with_neighborhood["GEOID"].astype(str).isin(check_geoids)
][["GEOID", "neighborhood_name"]]

print(result)

In [ ]:
missing = result[result["neighborhood_name"].isna()]
print("Missing neighborhood mapping:")
print(missing)